In [0]:
import os
import sys

import mlflow
import pandas as pd
from mlflow.models import infer_signature

sys.path.insert(0, os.path.abspath(".."))
from modelos import pronostico as P

NOMBRE = "vigia.ml.pronostico_dengue"  # catálogo.esquema.modelo
usuario = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/vigia-dengue-modelo")
mlflow.set_registry_uri("databricks-uc")  # los modelos se registran en Unity Catalog

In [0]:
elegida = spark.table("vigia.ml.backtest_resumen").where("elegida").first()
campeon, subida = elegida["modelo"], elegida["subida"]
variables = P.VARIABLES if campeon == "gbm_global" else [v for v in P.VARIABLES if v not in P.VARIABLES_CLIMA]
print(f"Campeón: {campeon}; regla: subida {subida:g}; error medio en el backtest: {elegida['mae_medio']:.2f} casos")

In [0]:
f = P.preparar(spark.table("vigia.gold.features_provincia_semana").toPandas())
hoy = f["inicio_semana"].max()
datos = P.entrenamiento(f, hoy)
modelo = P.nuevo_modelo(variables).fit(datos[variables], datos[P.ETIQUETAS])
firma = infer_signature(datos[variables], modelo.predict(datos[variables]))  # Unity Catalog exige la firma

# MLflow 3 llama `name` a lo que MLflow 2 llamaba `artifact_path`
donde = {"name": "modelo"} if int(mlflow.__version__.split(".")[0]) >= 3 else {"artifact_path": "modelo"}
with mlflow.start_run(run_name="campeon"):
    mlflow.log_params({"modelo": campeon, "subida": subida, "filas": len(datos),
                       "entrenado_hasta": str(datos["inicio_semana"].max().date()), **P.HIPERPARAMETROS})
    mlflow.log_metrics({k: float(elegida[k]) for k in ("mae_h1", "mae_h2", "mae_h3", "mae_h4", "mae_medio",
                                                       "sensibilidad", "falsas_por_anio", "anticipacion_costa")
                        if elegida[k] is not None})
    info = mlflow.sklearn.log_model(modelo, **donde, signature=firma, registered_model_name=NOMBRE)

cliente = mlflow.MlflowClient()
version = info.registered_model_version
cliente.set_registered_model_alias(NOMBRE, "champion", version)
cliente.set_model_version_tag(NOMBRE, version, "subida", str(subida))   # la regla de alerta va con el modelo
print(f"{NOMBRE}, versión {version}, con el alias champion")

In [0]:
cargado = mlflow.sklearn.load_model(f"models:/{NOMBRE}@champion")
ultima = f[f["inicio_semana"] == hoy]
pronostico = pd.DataFrame(cargado.predict(ultima[variables]).round(1), index=ultima.index,
                          columns=[f"casos_h{h}" for h in P.HORIZONTES])
display(pd.concat([ultima[["provincia", "anio_epi", "semana_epi", "casos_media_4s"]], pronostico], axis=1))